## Exercise — Judging a fit: R-squared, error and residuals

### Task 1 — Write MSE, RMSE and MAE

Here are the measured and predicted fractions absorbed (%) for five compounds:

```py
measured_pct = np.array([20, 45, 70, 85, 90])
pred_pct = np.array([26, 42, 64, 88, 95])
```

The three error metrics are:

$$\text{MSE} = \frac{1}{n}\sum_i (y_i - \hat{y}_i)^2 \qquad \text{RMSE} = \sqrt{\text{MSE}} \qquad \text{MAE} = \frac{1}{n}\sum_i |y_i - \hat{y}_i|$$

1. Import NumPy as `np`.
2. Write each formula as a short function, `mse`, `rmse` and `mae`, each taking `y_true` and `y_pred`. Let `rmse` call `mse`.
3. Print the three values for the arrays above, rounded to 2 decimals.


In [ ]:
import numpy as np

measured_pct = np.array([20, 45, 70, 85, 90])
pred_pct = np.array([26, 42, 64, 88, 95])

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

print("MSE:", round(mse(measured_pct, pred_pct), 2))
print("RMSE:", round(rmse(measured_pct, pred_pct), 2))
print("MAE:", round(mae(measured_pct, pred_pct), 2))

### Task 2 — Units, and one bad value

Below are two sets of predictions for five compounds. The first misses every compound by 3 %. The second is perfect except for the last compound, where a mistyped value gives a 15 % miss.

```py
measured_pct = np.array([30, 45, 60, 75, 90])
pred_even = np.array([33, 42, 63, 72, 93])
pred_one_miss = np.array([30, 45, 60, 75, 105])
print("even -> MSE (%^2):", round(mse(measured_pct, pred_even), 2), "RMSE (%):", round(rmse(measured_pct, pred_even), 2), "MAE (%):", round(mae(measured_pct, pred_even), 2))
print("one miss -> MSE (%^2):", round(mse(measured_pct, pred_one_miss), 2), "RMSE (%):", round(rmse(measured_pct, pred_one_miss), 2), "MAE (%):", round(mae(measured_pct, pred_one_miss), 2))
```

1. **Before running anything**, write down the six numbers these two lines print. Check that the unit in each label is correct.
2. Run the lines and compare with your prediction.
3. The 105 in `pred_one_miss` came from a typo. You do not want it to decide the verdict. Add one line that prints only the metric you would report for `pred_one_miss`, with its unit in the label.


In [ ]:
measured_pct = np.array([30, 45, 60, 75, 90])
pred_even = np.array([33, 42, 63, 72, 93])
pred_one_miss = np.array([30, 45, 60, 75, 105])
print("even -> MSE (%^2):", round(mse(measured_pct, pred_even), 2), "RMSE (%):", round(rmse(measured_pct, pred_even), 2), "MAE (%):", round(mae(measured_pct, pred_even), 2))
print("one miss -> MSE (%^2):", round(mse(measured_pct, pred_one_miss), 2), "RMSE (%):", round(rmse(measured_pct, pred_one_miss), 2), "MAE (%):", round(mae(measured_pct, pred_one_miss), 2))
print("fairer summary, MAE (%):", round(mae(measured_pct, pred_one_miss), 2))

### Task 3 — Fit a line and score it with R-squared

Ten compounds have a logP and a fraction of an oral dose absorbed (illustrative values):

| logP | -0.5 | 0.0 | 0.5 | 1.0 | 1.5 | 2.0 | 2.5 | 3.0 | 3.5 | 4.0 |
|---|---|---|---|---|---|---|---|---|---|---|
| fa_pct | 15 | 26 | 40 | 55 | 68 | 78 | 85 | 88 | 89 | 87 |

You are given this R-squared function. It implements $R^2 = 1 - SS_{res}/SS_{tot}$:

```py
def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot
```

1. Put the table into a DataFrame `df` with columns `logp` and `fa_pct`.
2. Fit the straight-line model `line(logp, slope, intercept)` with `curve_fit`, exactly as in the previous video. Store the predictions in `fa_pred`.
3. **Before running**, write down the R-squared you expect for two models: one that predicts the mean absorption for every compound, and one that predicts the measured values themselves.
4. Print the R-squared of the straight line, of the predict-the-mean model and of the perfect model, each rounded to 3 decimals.


In [ ]:
import pandas as pd
from scipy.optimize import curve_fit

df = pd.DataFrame({
    "logp": [-0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0],
    "fa_pct": [15, 26, 40, 55, 68, 78, 85, 88, 89, 87],
})

def line(logp, slope, intercept):
    return slope * logp + intercept

popt, pcov = curve_fit(line, df["logp"], df["fa_pct"])
slope, intercept = popt
fa_pred = line(df["logp"], slope, intercept)

def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

mean_pred = np.ones(len(df)) * np.mean(df["fa_pct"])
print("R2 straight line:", round(r_squared(df["fa_pct"], fa_pred), 3))
print("R2 predict the mean:", round(r_squared(df["fa_pct"], mean_pred), 3))
print("R2 perfect:", round(r_squared(df["fa_pct"], df["fa_pct"]), 3))

### Task 4 — Plot measured against predicted

Using `df` and `fa_pred` from Task 3, import `matplotlib.pyplot as plt` and draw a measured-vs-predicted plot:

- scatter the measured `fa_pct` (x-axis) against `fa_pred` (y-axis);
- draw the ideal line, predicted = measured, from 0 to 110 %, labelled `"ideal: predicted = measured"`, and show the legend;
- label both axes with units and give the figure a title, to the standard from track 2 video 8;
- save it as `measured_vs_predicted.png` with `dpi=300`, as you did in track 2 video 8, before showing it.

Look at where the lowest and highest compounds sit relative to the diagonal.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))
plt.scatter(df["fa_pct"], fa_pred)
plt.plot([0, 110], [0, 110], label="ideal: predicted = measured")
plt.xlabel("Measured fraction absorbed (%)")
plt.ylabel("Predicted fraction absorbed (%)")
plt.title("Measured vs predicted, straight-line model")
plt.legend()
plt.grid(True)
plt.savefig("measured_vs_predicted.png", dpi=300)
plt.show()

### Task 5 — Fix the residuals, then read them

A colleague wrote this and concluded that the line *over*-predicts the moderately lipophilic compounds in the middle of the range:

```py
residual_pct = fa_pred - df["fa_pct"]
```

The sign is the wrong way round, so the conclusion is reversed.

1. Write the corrected line for `residual_pct`.
2. Print the residuals rounded to 1 decimal.
3. Scatter `residual_pct` against `logp`, add a flat reference line at zero from logP −1.0 to 4.5, and label both axes, with units where there are any, and give the figure a title.

Check the signs at both ends and in the middle. If they form an arch rather than random scatter, the straight line is the wrong model, whatever its R-squared from Task 3 says.


In [ ]:
residual_pct = df["fa_pct"] - fa_pred
print(np.round(residual_pct, 1))
plt.figure(figsize=(8, 5))
plt.scatter(df["logp"], residual_pct)
plt.plot([-1.0, 4.5], [0, 0])
plt.xlabel("logP")
plt.ylabel("Residual, measured - predicted (%)")
plt.title("Residuals of the straight-line model")
plt.grid(True)
plt.show()